In [1]:
import os
os.environ["GDAL_PAM_ENABLED"] = 'NO'
import numpy as np
from glob import glob
from spectral.io import envi
from spectral.io.envi import read_envi_header, write_envi_header
import matplotlib.pyplot as plt
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.windows import from_bounds, Window
from shapely.geometry import box
from scipy.ndimage import gaussian_filter

from isofit.core.common import envi_header

import sys
sys.path.append('/store/carroll/repos/neon-isofit')
from utilities import *

In [2]:
home = '/store/carroll/col/data/2018/'
data = os.path.join(home, 'raw/rmbl/')

fp_skyview = os.path.join(home, 'skyview', 'sky_view_factor_10m')
rcc_folder = os.path.join(home, 'rccs/modtran')

out_folder = os.path.join(home, 'test_6c', 'subsets')

In [4]:
# just get this ready for the two little test regions 

rs = ['336000_4307000', '336000_4310000']

for r in rs:
    fp_ref = os.path.join(home, 'test_6c', 'subsets', f'NIS01_20180612_155442_rdn_ort_{r}')
    fp_out = os.path.join(home, 'test_6c', 'subsets', f'NIS01_20180612_155442_skyview_{r}')
    clip_skyview_per_flightline(fp_skyview, fp_ref, fp_out)

In [18]:
# now prepare rcc regions

suffixes = ['rdn_ort', 'rdn_obs_ort', 'rdn_ort_igm_ort'] # rdn, obs, loc

rs = ['329000_4307000', '329000_4307000',
      '332000_4302000', '332000_4302000',
      '318000_4303000', '318000_4303000']

flights = ['NIS01_20180626_165806', 'NIS01_20180619_155226',
           'NIS01_20180625_163511', 'NIS01_20180612_173258',
           'NIS01_20180620_171204', 'NIS01_20180620_170133']

for r_, flight in zip(rs, flights):
    print(r_, flight)
    fps = [x for x in glob(data+f'*/{flight}*') if x.endswith(tuple(suffixes))]
    fp_r = glob(home+f'raw/L3/discreteLidar/DTM/*{r_}*.tif')[0]
    for fp in fps:
        with rasterio.open(fp) as src, rasterio.open(fp_r) as r:
            subset_bounds = box(*src.bounds).intersection(box(*r.bounds)) # get intersection of bounds
            window = rasterio.windows.from_bounds(*subset_bounds.bounds, transform=src.transform)
            subset = src.read(window=window)
            
            # copy envi hdr data
            tags = src.tags(ns='ENVI').copy()
            for k in ['samples', 'lines', 'bands']:
                tags.pop(k, None)
                
            profile = src.profile.copy()
            profile.update({
                'height': subset.shape[1],
                'width': subset.shape[2],
                'transform': rasterio.windows.transform(window, src.transform),
                'interleave': tags['interleave']})
            
            fp_out = os.path.join(out_folder, f'{fp.split('/')[-1]}_{r_}')
            with rasterio.open(fp_out, 'w', **profile) as dst:
                dst.write(subset)
                dst.update_tags(ns='ENVI', **tags)
                try:
                    bn = tags['band_names']
                    bn = [s.strip() for s in bn[1:-1].split(',')]
                    for i, desc in enumerate(bn[:src.count], start=1):
                        dst.set_band_description(i, desc)
                except: pass

329000_4307000 NIS01_20180626_165806
329000_4307000 NIS01_20180619_155226
332000_4302000 NIS01_20180625_163511
332000_4302000 NIS01_20180612_173258
318000_4303000 NIS01_20180620_171204
318000_4303000 NIS01_20180620_170133


In [3]:
skyview_factor = '/store/carroll/col/data/2018/test_6c/subsets/NIS01_20180612_155442_skyview_336000_4310000.hdr'

svf_dataset = envi.open(envi_header(skyview_factor), skyview_factor)
svf_size = (svf_dataset.shape[0], svf_dataset.shape[1])
svf_max = np.nanmax(svf_dataset)

In [ ]:
# subset existing rdn, obs, loc files to the two NEON L3 lidar regions

for r_ in [r1, r2]:
    fp_r = glob(home+f'raw/L3/discreteLidar/CHM/*{r_}*.tif')[0]
    for fp in fps:
        with rasterio.open(fp) as src, rasterio.open(fp_r) as r:
            subset_bounds = box(*src.bounds).intersection(box(*r.bounds)) # get intersection of bounds
            window = rasterio.windows.from_bounds(*subset_bounds.bounds, transform=src.transform)
            subset = src.read(window=window)
            
            # copy envi hdr data
            tags = src.tags(ns='ENVI').copy()
            for k in ['samples', 'lines', 'bands']:
                tags.pop(k, None)
                
            profile = src.profile.copy()
            profile.update({
                'height': subset.shape[1],
                'width': subset.shape[2],
                'transform': rasterio.windows.transform(window, src.transform),
                'interleave': tags['interleave']})
            
            fp_out = os.path.join(out, f'{fp.split('/')[-1]}_{r_}')
            with rasterio.open(fp_out, 'w', **profile) as dst:
                dst.write(subset)
                dst.update_tags(ns='ENVI', **tags)
                try:
                    bn = tags['band_names']
                    bn = [s.strip() for s in bn[1:-1].split(',')]
                    for i, desc in enumerate(bn[:src.count], start=1):
                        dst.set_band_description(i, desc)
                except: pass